In [ ]:
import sys
import pickle
import numpy as np
import torch
from collections import defaultdict
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertModel

sys.path.insert(0, '/Users/jl/CMU-MultimodalSDK')
sys.path.insert(0, '/Users/jl/MISA/src')
from mmsdk import mmdatasdk as md
from create_dataset import word2id, return_unk

DATA_PATH = '/Users/jl/MISA/datasets/MOSEI'

DATASET = md.cmu_mosei
train_split = set(DATASET.standard_folds.standard_train_fold)
dev_split   = set(DATASET.standard_folds.standard_valid_fold)
test_split  = set(DATASET.standard_folds.standard_test_fold)

labels   = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_LabelsSentiment.csd.bak', DATA_PATH)
words_cs = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_TimestampedWords.csd.bak', DATA_PATH)
visual   = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_VisualFacet42.csd.bak', DATA_PATH)
acoustic = md.computational_sequence(DATA_PATH + '/CMU_MOSEI_COVAREP.csd.bak', DATA_PATH)

EPS = 1e-6
train, dev, test = [], [], []
MAX_TRAIN, MAX_DEV, MAX_TEST = 2000, 300, 500

print("Building dataset...")
for vid in list(labels.data.keys()):
    vid_id = vid.split('[')[0] if '[' in vid else vid
    if vid_id not in train_split and vid_id not in dev_split and vid_id not in test_split:
        continue
    try:
        label_feat = labels.data[vid]['features']
        word_feat  = words_cs.data[vid]['features']
        vis_feat   = visual.data[vid]['features']
        acou_feat  = acoustic.data[vid]['features']
    except:
        continue
    if len(word_feat) == 0 or len(vis_feat) == 0 or len(acou_feat) == 0:
        continue
    min_len = min(len(word_feat), len(vis_feat), len(acou_feat))
    word_feat = word_feat[:min_len]
    vis_feat  = vis_feat[:min_len]
    acou_feat = acou_feat[:min_len]
    actual_words, word_ids, vis_list, acou_list = [], [], [], []
    for i, word in enumerate(word_feat):
        try:
            w = word[0].decode('utf-8') if isinstance(word[0], bytes) else str(word[0])
        except:
            continue
        if w != 'sp':
            actual_words.append(w)
            word_ids.append(word2id[w])
            vis_list.append(vis_feat[i])
            acou_list.append(acou_feat[i])
    if len(word_ids) == 0:
        continue

    words_arr = np.asarray(word_ids)
    vis_arr   = np.nan_to_num(np.asarray(vis_list))
    acou_arr  = np.nan_to_num(np.asarray(acou_list))

    # Average across annotators, keep all 7 values
    # [sentiment, happiness, sadness, anger, fear, disgust, surprise]
    if label_feat.shape[1] >= 7:
        label_arr = np.nanmean(label_feat, axis=0, keepdims=True)  # (1, 7)
    else:
        label_arr = np.array([[np.nanmean(label_feat[:, 0]), 0, 0, 0, 0, 0, 0]])

    vis_arr  = np.nan_to_num((vis_arr  - vis_arr.mean(0,  keepdims=True)) / (EPS + vis_arr.std(0,  keepdims=True)))
    acou_arr = np.nan_to_num((acou_arr - acou_arr.mean(0, keepdims=True)) / (EPS + acou_arr.std(0, keepdims=True)))

    entry = ((words_arr, vis_arr, acou_arr, actual_words), label_arr, vid)

    if vid_id in train_split and len(train) < MAX_TRAIN:
        train.append(entry)
    elif vid_id in dev_split and len(dev) < MAX_DEV:
        dev.append(entry)
    elif vid_id in test_split and len(test) < MAX_TEST:
        test.append(entry)
    if len(train) >= MAX_TRAIN and len(dev) >= MAX_DEV and len(test) >= MAX_TEST:
        break

print(f"Train: {len(train)}, Dev: {len(dev)}, Test: {len(test)}")
print(f"Label shape: {train[0][1].shape}")
print(f"Sample label: {train[0][1]}")

for split, data in [('train', train), ('dev', dev), ('test', test)]:
    with open(f'{DATA_PATH}/{split}.pkl', 'wb') as f:
        pickle.dump(data, f)
print("Base pickles saved.")

[2026-05-16 00:34:48.639] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_LabelsSentiment.csd.bak ...
[2026-05-16 00:34:48.737] | Status  | Checking the integrity of the <All Labels> computational sequence ...
[2026-05-16 00:34:48.737] | Status  | Checking the format of the data in <All Labels> computational sequence ...


[2026-05-16 00:34:49.342] | Success | <All Labels> computational sequence data in correct format.
[2026-05-16 00:34:49.342] | Status  | Checking the format of the metadata in <All Labels> computational sequence ...
[2026-05-16 00:34:49.342] | Warning | <All Labels> computational sequence does not have all the required metadata ... continuing 
[2026-05-16 00:34:49.350] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_TimestampedWords.csd.bak ...
[2026-05-16 00:34:49.383] | Status  | Checking the integrity of the <words> computational sequence ...
[2026-05-16 00:34:49.383] | Status  | Checking the format of the data in <words> computational sequence ...


[2026-05-16 00:34:49.971] | Success | <words> computational sequence data in correct format.
[2026-05-16 00:34:49.971] | Status  | Checking the format of the metadata in <words> computational sequence ...
[2026-05-16 00:34:49.971] | Warning | <words> computational sequence does not have all the required metadata ... continuing 
[2026-05-16 00:34:49.981] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_VisualFacet42.csd.bak ...
[2026-05-16 00:34:50.014] | Status  | Checking the integrity of the <OpenFace_2> computational sequence ...
[2026-05-16 00:34:50.014] | Status  | Checking the format of the data in <OpenFace_2> computational sequence ...


[2026-05-16 00:34:50.474] | Success | <OpenFace_2> computational sequence data in correct format.
[2026-05-16 00:34:50.474] | Status  | Checking the format of the metadata in <OpenFace_2> computational sequence ...
[2026-05-16 00:34:50.474] | Warning | <OpenFace_2> computational sequence does not have all the required metadata ... continuing 
[2026-05-16 00:34:50.485] | Success | Computational sequence read from file /Users/jl/MISA/datasets/MOSEI/CMU_MOSEI_COVAREP.csd.bak ...
[2026-05-16 00:34:50.980] | Status  | Checking the integrity of the <COVAREP> computational sequence ...
[2026-05-16 00:34:50.980] | Status  | Checking the format of the data in <COVAREP> computational sequence ...


/var/folders/x5/v5smrj3d2mv36k9vry639c9c0000gn/T/ipykernel_31734/2421202439.py:69: RuntimeWarning: invalid value encountered in divide
  acou_arr = np.nan_to_num((acou_arr - acou_arr.mean(0, keepdims=True)) / (EPS + acou_arr.std(0, keepdims=True)))


[2026-05-16 00:34:51.472] | Success | <COVAREP> computational sequence data in correct format.
[2026-05-16 00:34:51.472] | Status  | Checking the format of the metadata in <COVAREP> computational sequence ...
[2026-05-16 00:34:51.472] | Warning | <COVAREP> computational sequence does not have all the required metadata ... continuing 
Building dataset...
Train: 2249, Dev: 300, Test: 676
Base pickles saved.


In [12]:
print("Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model.eval()

def extract_bert_features(actual_words, max_len=64):
    text = ' '.join(actual_words)
    tokens = tokenizer(text, max_length=max_len, padding='max_length',
                       truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state[:, 0, :].squeeze().numpy()

for split in ['train', 'dev', 'test']:
    print(f"\nProcessing {split}...")
    with open(f'{DATA_PATH}/{split}.pkl', 'rb') as f:
        data = pickle.load(f)
    enriched = []
    for (words, vis, acou, actual_words), label, vid in tqdm(data):
        bert_feat = extract_bert_features(actual_words)
        enriched.append(((words, vis, acou, actual_words, bert_feat), label, vid))
    with open(f'{DATA_PATH}/{split}_bert.pkl', 'wb') as f:
        pickle.dump(enriched, f)
    print(f"Saved {split}_bert.pkl — {len(enriched)} samples")

del bert_model
print("\nDone!")

Loading DistilBERT...


Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



Processing train...


100%|██████████| 2249/2249 [00:49<00:00, 45.64it/s]


Saved train_bert.pkl — 2249 samples

Processing dev...


100%|██████████| 300/300 [00:06<00:00, 45.82it/s]


Saved dev_bert.pkl — 300 samples

Processing test...


100%|██████████| 676/676 [00:15<00:00, 44.33it/s]


Saved test_bert.pkl — 676 samples

Done!
